In [ ]:
# import the required libraries
import pandas as pd
from transformers import GPT2Tokenizer, GPT2Model
import torch
import numpy as np

# read the input data (cleaned embedded keywords-title dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the cleaned embedded keywords-title dataset is the dataset 
formed by merging & cleaning the embedded keywords and embedded title dataset.
"""
movies_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/gpt2_chunk_1.csv")

In [ ]:
movies_df.head(10)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...
1,False,1115,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...
2,False,1116,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",-1.05129234e-01 1.31294459e-01 -1.72948554e-0...,-4.49490517e-01 -5.11738248e-02 7.86833838e-0...
3,False,1123,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-2.57868499e-01 6.26966059e-02 -7.99108744e-0...,2.59599864e-01 1.75047010e-01 -2.48356387e-0...
4,False,1124,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",2.70989593e-02 1.46407694e-01 -3.80637795e-0...,8.48557949e-02 1.20956279e-01 1.18412793e-0...
5,False,1125,"Drama, Music",tt0443489,en,Dreamgirls,A trio of female soul singers cross over to th...,28.994,12/25/2006,English,6.750,947.0,1125.0,music record manager show business drug addict...,"[8, 18]",-9.32795629e-02 1.51976109e-01 -3.74815643e-0...,-2.23533288e-02 -4.14516144e-02 5.20074368e-0...
6,False,1126,"Drama, Mystery, Thriller",tt0385017,en,In My Father's Den,"Paul (Macfadyen), a prize-winning war journali...",8.440,10/7/2004,English,7.033,90.0,1126.0,sibling relationship soulmates funeral loss of...,"[8, 7, 10]",1.38275236e-01 6.01867028e-03 -3.08690369e-0...,-2.28183910e-01 -1.40631601e-01 -1.98941201e-0...
7,False,1127,Drama,tt0434292,es,Princesas,Caye is a young prostitute whose family is una...,7.356,9/2/2005,Spanish,6.750,76.0,1127.0,future depression friendship prostitute rape s...,[8],1.66739821e-02 1.74580410e-01 -4.18109119e-0...,-3.43883783e-01 -1.48477644e-01 -8.37202072e-0...
8,False,1162,Drama,tt0056241,en,The Miracle Worker,"The true story of the frightening, lonely worl...",9.404,5/23/1962,English,7.800,280.0,1162.0,blindness and impaired vision deaf-mute educat...,[8],2.34389082e-02 7.38373399e-03 -3.21811736e-0...,2.34131906e-02 -4.92307432e-02 3.73071134e-0...
9,False,1163,"Drama, TV Movie",tt0246786,en,The Miracle Worker,"Devoted teacher Anne Sullivan leads deaf, blin...",10.345,11/12/2000,English,6.900,31.0,1163.0,blindness and impaired vision education woman ...,"[8, 14]",-2.77143493e-02 -1.44032761e-02 -5.36748469e-0...,2.34131906e-02 -4.92307432e-02 3.73071134e-0...


In [ ]:
movies_df.shape[0]

46944

In [ ]:
movies_df.columns

Index(['adult', 'id', 'genres', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
drop_columns = ['adult', 'genres', 'original_language',
       'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview
0,1114,tt0373861,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...
1,1115,tt0419773,Gespenster,"Nina, an end-of-teenage orphan with mental pro..."
2,1116,tt0460989,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...
3,1123,tt0437232,Catch a Fire,The true story of anti-apartheid activists in ...
4,1124,tt0482571,The Prestige,A mysterious story of two magicians whose inte...


In [ ]:
# reduce the size of the overview column to 500 characters of string
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column '{}': {}".format('overview', max_length))
movies_df['overview'] = movies_df['overview'].str.slice(0, 500)
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column after strip '{}': {}".format('overview', max_length))

Maximum length of column 'overview': 1186.0
Maximum length of column after strip 'overview': 500.0


In [ ]:
# load the pre-trained gpt-2 tokenizer and model
model_name='gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2Model.from_pretrained(model_name)

"""
:function: create_word_embeddings
:param sentence: takes a input sentence to be embedded i.e., movie overview
:return: embeded sentence vector of size 768
""" 
def create_word_embeddings(sentence):
    if pd.isna(sentence):
      return np.zeros(768)
    else:
      input_ids = tokenizer.encode(sentence, return_tensors='pt')
      with torch.no_grad():
          output = model(input_ids)
      embeddings = output.last_hidden_state.mean(dim=1).squeeze().numpy()
      return embeddings

# Apply the function to each row and create a new column 'overview_embedding'
movies_df['overview_embedding'] = movies_df['overview'].apply(create_word_embeddings)

In [ ]:
movies_df['overview_embedding'] = movies_df['overview_embedding'].apply(lambda x: str(x))
movies_df.loc[0, 'overview_embedding']

'[-7.90620819e-02  1.27022997e-01 -7.71413565e-01 -9.57237408e-02\n -1.98667616e-01 -1.43106177e-01  1.47545946e+00 -1.46336094e-01\n  2.09725097e-01  2.01271579e-01  6.13774598e-01 -2.42962018e-01\n -1.20739460e-01 -9.45630744e-02 -2.46817078e-02  2.44118378e-01\n  2.24141732e-01  2.90502310e-02  2.40911886e-01 -1.83276698e-01\n  3.87563139e-01 -1.34887502e-01 -2.12923929e-01 -1.08384244e-01\n  2.81496137e-01  2.80038744e-01  1.71093139e-04  2.14782774e-01\n -3.69584337e-02  5.46785817e-02 -1.51102897e-02 -1.52639505e-02\n -1.12024054e-01  4.47068959e-02 -1.11463457e-01  3.52249593e-01\n  7.42878265e+01  9.69035327e-02 -1.12410896e-01 -8.03367421e-02\n -2.11347770e-02  1.09749874e-02 -2.69903690e-01 -8.47044773e-03\n -4.59584206e-01 -2.71406293e-01  1.03832766e-01  4.58366014e-02\n  3.67437042e-02  1.76635519e-01  1.79629937e-01  2.99487948e-01\n  2.18376800e-01  1.14052691e-01  1.47575065e-01  1.31625795e+00\n  2.09961370e-01  4.29361686e-02 -2.33544439e-01  2.52068430e-01\n -1.12367

In [ ]:
# convert the embedded column array to embedded string and clean the overview_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_df['overview_embedding_string'] = movies_df['overview_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '').str.replace(',', '')

In [ ]:
movies_df.loc[0, 'overview_embedding_string']

'-7.90620819e-02  1.27022997e-01 -7.71413565e-01 -9.57237408e-02 -1.98667616e-01 -1.43106177e-01  1.47545946e+00 -1.46336094e-01  2.09725097e-01  2.01271579e-01  6.13774598e-01 -2.42962018e-01 -1.20739460e-01 -9.45630744e-02 -2.46817078e-02  2.44118378e-01  2.24141732e-01  2.90502310e-02  2.40911886e-01 -1.83276698e-01  3.87563139e-01 -1.34887502e-01 -2.12923929e-01 -1.08384244e-01  2.81496137e-01  2.80038744e-01  1.71093139e-04  2.14782774e-01 -3.69584337e-02  5.46785817e-02 -1.51102897e-02 -1.52639505e-02 -1.12024054e-01  4.47068959e-02 -1.11463457e-01  3.52249593e-01  7.42878265e+01  9.69035327e-02 -1.12410896e-01 -8.03367421e-02 -2.11347770e-02  1.09749874e-02 -2.69903690e-01 -8.47044773e-03 -4.59584206e-01 -2.71406293e-01  1.03832766e-01  4.58366014e-02  3.67437042e-02  1.76635519e-01  1.79629937e-01  2.99487948e-01  2.18376800e-01  1.14052691e-01  1.47575065e-01  1.31625795e+00  2.09961370e-01  4.29361686e-02 -2.33544439e-01  2.52068430e-01 -1.12367988e-01 -4.61391695e-02  2.1014

In [ ]:
drop_columns = ['overview_embedding']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview,overview_embedding_string
0,1114,tt0373861,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,-7.90620819e-02 1.27022997e-01 -7.71413565e-0...
1,1115,tt0419773,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",-1.14440762e-01 -7.02230632e-02 -2.19043717e-0...
2,1116,tt0460989,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,-1.15996398e-01 -2.08327860e-01 -7.92207662e-0...
3,1123,tt0437232,Catch a Fire,The true story of anti-apartheid activists in ...,-2.46082172e-01 1.04989491e-01 -3.55010718e-0...
4,1124,tt0482571,The Prestige,A mysterious story of two magicians whose inte...,6.16751704e-03 -1.18230581e-01 -7.90594935e-0...


In [ ]:
movies_df['overview_embedding_string'] = movies_df['overview_embedding_string'].str.strip()

In [ ]:
movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/old_final_mkg_dataset_gpt2.csv")

In [ ]:
merged_df = pd.merge(movies_df, movies_keywords_genres, on=['id', 'imdb_id', 'original_title'], how='inner')

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,1114,tt0373861,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,-7.90620819e-02 1.27022997e-01 -7.71413565e-0...,False,"Documentary, Drama",mn,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...
1,1115,tt0419773,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",-1.14440762e-01 -7.02230632e-02 -2.19043717e-0...,False,Drama,de,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...
2,1116,tt0460989,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,-1.15996398e-01 -2.08327860e-01 -7.92207662e-0...,False,"Drama, War",en,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",-1.05129234e-01 1.31294459e-01 -1.72948554e-0...,-4.49490517e-01 -5.11738248e-02 7.86833838e-0...
3,1123,tt0437232,Catch a Fire,The true story of anti-apartheid activists in ...,-2.46082172e-01 1.04989491e-01 -3.55010718e-0...,False,"Action, Drama, Thriller",en,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-2.57868499e-01 6.26966059e-02 -7.99108744e-0...,2.59599864e-01 1.75047010e-01 -2.48356387e-0...
4,1124,tt0482571,The Prestige,A mysterious story of two magicians whose inte...,6.16751704e-03 -1.18230581e-01 -7.90594935e-01...,False,"Drama, Mystery, Science Fiction",en,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",2.70989593e-02 1.46407694e-01 -3.80637795e-0...,8.48557949e-02 1.20956279e-01 1.18412793e-0...


In [ ]:
merged_df.shape[0]

46944

In [ ]:
merged_df.columns

Index(['id', 'imdb_id', 'original_title', 'overview_x',
       'overview_embedding_string', 'adult', 'genres', 'original_language',
       'overview_y', 'popularity', 'release_date', 'spoken_languages',
       'vote_average', 'vote_count', 'movie_id', 'keywords', 'genres_indices',
       'keywords_embedding_string', 'original_title_embedding_string'],
      dtype='object')

In [ ]:
nan_count = merged_df['overview_embedding_string'].isnull().sum(); print(f"The column 'overview_embedding' has {nan_count} NaN values.")
nan_count = merged_df['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
merged_df.dropna(subset=['overview_embedding_string'], inplace=True)
merged_df.dropna(subset=['genres'], inplace=True)

The column 'overview_embedding' has 0 NaN values.
The column 'genres' has 0 NaN values.


In [ ]:
merged_df.shape[0]

46944

In [ ]:
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/final_mkg_dataset_gpt2_chunk_1.csv', index=False)

In [ ]:
#  Test

In [ ]:
movies_df.shape[0]

46944

In [ ]:
movies_keywords_genres.shape[0]

234719

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,1114,tt0373861,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,-7.90620819e-02 1.27022997e-01 -7.71413565e-0...,False,"Documentary, Drama",mn,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...
1,1115,tt0419773,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",-1.14440762e-01 -7.02230632e-02 -2.19043717e-0...,False,Drama,de,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...
2,1116,tt0460989,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,-1.15996398e-01 -2.08327860e-01 -7.92207662e-0...,False,"Drama, War",en,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",-1.05129234e-01 1.31294459e-01 -1.72948554e-0...,-4.49490517e-01 -5.11738248e-02 7.86833838e-0...
3,1123,tt0437232,Catch a Fire,The true story of anti-apartheid activists in ...,-2.46082172e-01 1.04989491e-01 -3.55010718e-0...,False,"Action, Drama, Thriller",en,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-2.57868499e-01 6.26966059e-02 -7.99108744e-0...,2.59599864e-01 1.75047010e-01 -2.48356387e-0...
4,1124,tt0482571,The Prestige,A mysterious story of two magicians whose inte...,6.16751704e-03 -1.18230581e-01 -7.90594935e-01...,False,"Drama, Mystery, Science Fiction",en,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",2.70989593e-02 1.46407694e-01 -3.80637795e-0...,8.48557949e-02 1.20956279e-01 1.18412793e-0...


In [ ]:
# merge the embedded keywords, title and overview columns to form the input gpt2 embedded dataset chunk-1
drop_columns = ['imdb_id', 'original_title', 'overview_x', 'adult', 'original_language', 'overview_y', 'popularity', 'spoken_languages', 'vote_average', 'vote_count', 'movie_id', 'keywords']
merged_df = merged_df.drop(columns=drop_columns)
merged_df.head(5)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,1114,-7.90620819e-02 1.27022997e-01 -7.71413565e-0...,"Documentary, Drama",6/29/2003,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...
1,1115,-1.14440762e-01 -7.02230632e-02 -2.19043717e-0...,Drama,9/15/2005,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...
2,1116,-1.15996398e-01 -2.08327860e-01 -7.92207662e-0...,"Drama, War",6/23/2006,"[8, 16]",-1.05129234e-01 1.31294459e-01 -1.72948554e-0...,-4.49490517e-01 -5.11738248e-02 7.86833838e-0...
3,1123,-2.46082172e-01 1.04989491e-01 -3.55010718e-0...,"Action, Drama, Thriller",10/27/2006,"[19, 8, 10]",-2.57868499e-01 6.26966059e-02 -7.99108744e-0...,2.59599864e-01 1.75047010e-01 -2.48356387e-0...
4,1124,6.16751704e-03 -1.18230581e-01 -7.90594935e-01...,"Drama, Mystery, Science Fiction",10/19/2006,"[8, 7, 4]",2.70989593e-02 1.46407694e-01 -3.80637795e-0...,8.48557949e-02 1.20956279e-01 1.18412793e-0...


In [ ]:
merged_df.shape[0]

46944

In [ ]:
# export the embedded keywords, title and overview dataset (chunk-1)
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_1.csv', index=False)

In [ ]:
# Testing read

In [ ]:
test_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_1.csv")

In [ ]:
test_df.shape[0]

46944

In [ ]:
test_df.columns

Index(['id', 'overview_embedding_string', 'genres', 'release_date',
       'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
test_df.head(4)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,1114,-7.90620819e-02 1.27022997e-01 -7.71413565e-0...,"Documentary, Drama",6/29/2003,"[11, 8]",7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...
1,1115,-1.14440762e-01 -7.02230632e-02 -2.19043717e-0...,Drama,9/15/2005,[8],9.63690951e-02 6.21698722e-02 -5.16188264e-0...,1.28780708e-01 1.27590522e-01 -6.06716909e-0...
2,1116,-1.15996398e-01 -2.08327860e-01 -7.92207662e-0...,"Drama, War",6/23/2006,"[8, 16]",-1.05129234e-01 1.31294459e-01 -1.72948554e-0...,-4.49490517e-01 -5.11738248e-02 7.86833838e-0...
3,1123,-2.46082172e-01 1.04989491e-01 -3.55010718e-0...,"Action, Drama, Thriller",10/27/2006,"[19, 8, 10]",-2.57868499e-01 6.26966059e-02 -7.99108744e-0...,2.59599864e-01 1.75047010e-01 -2.48356387e-0...
